# Supervisor가 Worker를 반복 조정하는 Multi-Agent

이 실습은 Researcher와 Coder를 Supervisor가 반복 선택하는 Graph를 직접 완성한다. 모든 코드셀은 비어 있으므로 위에서부터 설명과 흐름도를 읽고 필요한 객체를 단계별로 작성한다.

각 단계에서 입력이 어떤 State field로 들어오고, 결과가 다음 Node의 어떤 입력이 되는지 먼저 확인한다. 외부 모델과 검색은 마지막 실행 단계에서만 호출한다.

## 패키지 설치

LangChain Agent, OpenAI Chat Model, LangGraph와 Tavily 검색에 필요한 패키지 목록을 현재 Jupyter kernel에 설치한다. 출력은 뒤 셀에서 import할 수 있는 실행 환경이며, 설치 셀 자체는 API를 호출하지 않는다.

In [ ]:
%pip install -U langchain langchain-openai langgraph langchain-tavily python-dotenv

## 환경 변수와 모델 ID 준비

`find_dotenv`와 `load_dotenv`는 현재 작업 디렉터리의 환경 설정을 입력으로 읽는다. 출력은 두 Worker와 Supervisor가 공유할 모델 ID이며 뒤의 `ChatOpenAI` 생성에 사용한다. LangSmith 추적은 선택 기능으로 두고 비밀값은 출력하지 않는다.

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)

if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", "langgraph-supervision")

OPENAI_CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")

## 전체 부품 지도

먼저 제한된 Tool을 준비하고, Worker 결과를 누적할 State와 adapter를 만든다. 그다음 두 Worker와 Supervisor를 조립하고 Conditional Edge로 반복 경로를 완성한다.

```text
Tool → Worker Agent → Worker adapter → messages
                                     ↓
END ← FINISH ← Conditional Edge ← Supervisor
                 ↓
          Researcher 또는 Coder
```

Worker는 맡은 작업을 수행하고 Supervisor는 누적된 결과를 읽어 다음 Worker 또는 종료를 선택한다.

## Worker별 권한을 Tool로 제한하기

`TavilySearch`로 Researcher의 검색 Tool을 준비하고, `tool` decorator로 Coder가 사용할 평균 함수를 Tool로 바꾼다. 각 Tool의 입력과 반환값은 이후 `create_agent`의 권한 목록으로 전달된다.

### 제한된 평균 Tool의 순수 계산 확인

`calculate_average` Tool wrapper의 `func`에 숫자 목록을 입력해 외부 callback이나 모델 호출 없이 원래 Python 구현만 확인한다. 출력 실수는 이후 Coder가 사용하는 계산 근거이다.

## 상위 State와 Worker 결과 전달 형식

`TypedDict`로 누적 대화와 Supervisor가 선택한 다음 목적지를 갖는 State를 정의하고, `add_messages`를 messages reducer로 사용한다. Worker adapter는 State의 messages를 Agent에 전달하고 마지막 응답을 이름이 있는 `HumanMessage`로 바꿔 다음 Supervisor가 읽게 한다.

## Researcher와 Coder Worker 만들기

하나의 `ChatOpenAI` 모델을 공유하되 `create_agent`에 전달할 Tool과 역할 정책은 Worker별로 다르게 구성한다. 생성한 Agent와 Worker 이름을 `partial`로 adapter에 고정해 Graph가 호출할 두 Node로 바꾼다.

## Supervisor의 선택을 구조화하기

`Enum`과 Pydantic model로 Supervisor의 선택지를 Researcher, Coder와 FINISH로 제한한다. `ChatPromptTemplate`의 message placeholder에 누적 대화를 넣고 구조화된 선택 결과를 State의 next field로 전달한다.

## Worker가 Supervisor로 돌아오는 Graph 조립

`StateGraph`는 앞에서 만든 State schema, Supervisor와 두 Worker Node를 입력으로 받는다. START는 Supervisor로 들어가고 Worker는 실행 뒤 Supervisor로 돌아오며, Conditional Edge는 next 값을 실제 Worker 또는 END에 연결한다. `compile`의 출력 graph는 뒤의 route 확인과 전체 실행 셀에서 사용한다.

## 저장된 `next`가 실제 분기값이 되는지 확인하기

모델을 호출하지 않는 최소 State를 `route_next`에 전달해 반환값을 확인한다. 반환 문자열은 앞에서 등록한 Conditional Edge가 다음 Node를 선택할 때 사용한다.

## 한 번의 요청에서 선택과 종료 추적하기

검색과 평균 계산을 함께 요구하는 사용자 message를 Graph에 전달한다. `stream`의 Node별 update에서 Supervisor의 next와 Worker 이름을 모아 `Supervisor → Worker → Supervisor → FINISH` 순서를 확인한다.

## 정리

완성된 Graph에서 Worker는 제한된 Tool로 전문 작업을 수행하고, Supervisor는 누적 messages를 근거로 다음 경로를 고른다. 마지막 trace에서 Worker 결과가 다시 Supervisor로 돌아온 뒤 FINISH에 도달하는지 점검한다.